# Schema Design Patterns

## Overview
MongoDB's flexible document model enables schema designs impossible in SQL -- but it also means schema design decisions have a much larger performance impact. The core question is always: **embed or reference?**

**The embed vs reference decision:**

| Factor | Embed | Reference |
|---|---|---|
| Access pattern | Always accessed together | Accessed independently |
| Relationship cardinality | One-to-few | One-to-many or many-to-many |
| Data ownership | Belongs to one parent | Shared across multiple parents |
| Update frequency | Rarely updated | Frequently updated independently |
| Size | Bounded (stays < 1 MB) | Unbounded (can grow indefinitely) |

**General guidance:**
- Start with embedding -- it gives the best read performance
- Switch to referencing when documents get large, when the sub-data is accessed independently, or when the relationship is many-to-many
- The 16 MB BSON document limit is a hard ceiling for embedded documents

**MongoDB vs relational normalisation:**
SQL normalisation avoids data duplication for integrity. MongoDB often **intentionally duplicates** data (denormalises) to improve read performance -- but duplication requires careful update management. This is a deliberate tradeoff, not a mistake.

---

In [ ]:

print("=== Embedding vs Referencing ===")
embedding = '''
# ── EMBEDDING: store related data inside the parent document ─────
# Use when: data is always accessed together; sub-data belongs to
#            one parent only; sub-data is bounded in size.

{
    "patient_id": "P001",
    "first_name": "Aroha",
    "last_name":  "Ngata",
    "conditions": ["hypertension", "type2_diabetes"],   # array of scalars
    "contact": {                                         # embedded subdocument
        "email": "aroha@email.com",
        "phone": "506-555-0101"
    },
    "latest_vitals": {                                   # embed the most recent
        "recorded_at": "2023-10-01",
        "bp_systolic": 138,
        "bp_diastolic": 88,
        "weight_kg": 74.5
    }
}
# Pro: single document read returns all data; fast; atomic updates
# Con: document grows; array can hit 16 MB BSON limit; hard to query sub-data alone
'''
print("Embedding pattern:")
print(embedding)

referencing = '''
# ── REFERENCING: store related data in a separate collection ─────
# Use when: sub-data is large or unbounded; accessed independently;
#            shared across multiple parents.

# patients collection:
{"patient_id": "P001", "first_name": "Aroha", ...}

# encounters collection (references patient by ID):
{"encounter_id": "E001", "patient_id": "P001",   # reference
 "encounter_date": "2023-04-10", "encounter_type": "outpatient"}
{"encounter_id": "E002", "patient_id": "P001",
 "encounter_date": "2023-07-15", "encounter_type": "inpatient"}

# To query: use $lookup (see aggregation_pipeline notebook)
# Pro: encounters collection is independently queryable; no size limit
# Con: requires $lookup (extra read); not atomic across collections
'''
print("Referencing pattern:")
print(referencing)


---
## Common schema patterns

In [ ]:

print("=== Common schema patterns ===")
patterns = [
    ("Polymorphic",
     "Documents in the same collection have different shapes based on a type field.",
     '''
# All clinical observations in one collection; type determines shape
{"obs_type": "lab", "patient_id": "P001", "test": "HbA1c", "value": 7.2, "unit": "%"}
{"obs_type": "vital", "patient_id": "P001", "bp_systolic": 138, "weight_kg": 74.5}
{"obs_type": "imaging", "patient_id": "P001", "modality": "MRI", "body_part": "brain"}
# Use a discriminator field (obs_type) and partial indexes per type
'''),
    ("Bucket / Roll-up",
     "Pre-aggregate time-series data into buckets to limit document count.",
     '''
# Instead of one doc per sensor reading (millions per day):
{"sensor_id": "S01", "timestamp": "2023-10-01T14:32:00", "temp_c": 22.4}

# Bucket one document per sensor per hour:
{"sensor_id": "S01", "hour": "2023-10-01T14:00",
 "readings": [22.4, 22.5, 22.3, ...],    # up to 60 readings
 "count": 60, "min": 22.1, "max": 22.7, "avg": 22.4}
# Dramatically reduces document count; efficient for time-range queries
'''),
    ("Outlier",
     "Keep common data embedded; overflow rare large data to a linked document.",
     '''
# Most posts have < 100 comments → embed them
{"post_id": "post_01", "comments": [...], "has_extra_comments": False}

# Viral post with 50,000 comments → overflow to separate collection
{"post_id": "post_viral", "comments": [...first 100...],
 "has_extra_comments": True}
# Separate overflow:
{"post_id": "post_viral", "page": 2, "comments": [...next 100...]}
'''),
    ("Extended Reference",
     "Embed frequently-read fields from the referenced document to avoid $lookup.",
     '''
# orders collection embeds the fields most often needed from customers:
{"order_id": "O001", "customer_id": "C001",
 "customer_name": "Aroha Ngata",          # duplicated from customers
 "customer_email": "aroha@email.com",     # duplicated from customers
 "order_total": 149.99, "status": "shipped"}
# Avoids $lookup for the common "show order + customer name" query
# Con: must update duplicated fields when customer changes name/email
'''),
]
for name, desc, example in patterns:
    print(f"  Pattern: {name}")
    print(f"  When to use: {desc}")
    print(f"  Example:{example}")


---
## Schema validation with $jsonSchema

In [ ]:

print("=== Schema validation with $jsonSchema ===")
validation = '''
# Enforce a schema on a collection (MongoDB 3.6+)
db.create_collection("patients", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["patient_id", "first_name", "last_name", "sex", "dob"],
        "properties": {
            "patient_id": {"bsonType": "string",  "description": "Required string"},
            "first_name":  {"bsonType": "string"},
            "last_name":   {"bsonType": "string"},
            "sex":         {"bsonType": "string",  "enum": ["M", "F", "Other"]},
            "dob":         {"bsonType": "string",  "pattern": "^\\d{4}-\\d{2}-\\d{2}$"},
            "conditions":  {"bsonType": "array",
                            "items": {"bsonType": "string"}},
            "contact":     {"bsonType": "object",
                            "properties": {
                                "email": {"bsonType": "string"},
                                "phone": {"bsonType": "string"}
                            }}
        }
    }
})

# Attempts to insert documents that violate the schema raise WriteError.
# Existing documents are NOT retroactively validated.
# Add to existing collection:
db.command("collMod", "patients",
           validator={"$jsonSchema": {...}},
           validationLevel="moderate")   # moderate=only validate updates; strict=all writes
'''
print(validation)

print("When to use schema validation:")
use_cases = [
    "Required fields enforcement (required: [...]) -- catches application-layer bugs",
    "Enum constraints on status/type fields (enum: ['active', 'inactive'])",
    "Type safety for numeric fields (bsonType: 'double' prevents accidental string insert)",
    "Regulatory/compliance environments requiring data integrity guarantees",
    "NOT needed for exploratory/prototype collections where schema is still evolving",
]
for u in use_cases:
    print(f"  - {u}")


---
## Common Pitfalls

**1. Unbounded embedding causing documents to exceed 16 MB**
Embedding every encounter inside a patient document is fine for 10 encounters, but after years of care a patient may have thousands. BSON documents are hard-capped at 16 MB. Use referencing for data that grows over time, and embed only data that is bounded and accessed together.

**2. Referencing without understanding the $lookup cost**
Normalising everything into separate collections and joining with `$lookup` works, but every `$lookup` is an extra read from another collection. MongoDB does not have foreign key indexes automatically -- ensure the join field is indexed in the foreign collection, otherwise `$lookup` performs a full collection scan per document.

**3. Duplicating data in extended reference pattern without an update strategy**
Embedding `customer_name` in orders avoids a join, but when a customer changes their name, every order document must be updated. If the update fails partway through, data is inconsistent. Only embed fields that are immutable (like a customer ID) or rarely change, and have a clear re-sync strategy for fields that do change.

**4. No discriminator field in polymorphic collections**
Storing mixed document types without a `type` or `obs_type` field makes it impossible to query only lab results vs vitals vs imaging. Always include a discriminator field and create a partial index per type: `create_index('test', partialFilterExpression={'obs_type': 'lab'})`.

**5. Adding schema validation to existing collections without checking existing data**
`validationLevel: 'strict'` rejects any update to a document that does not comply with the new schema. If existing documents have missing required fields, even unrelated updates to those documents will fail. Always audit existing data before enforcing strict validation on a live collection.

**6. Bucket pattern bucket overflow**
A bucket document pre-aggregates N readings. If a burst of data pushes a bucket over the limit (e.g. more than 60 readings in an hour), the bucket document grows unboundedly unless you implement overflow handling. Cap bucket size with a counter and create a new bucket when the limit is reached.


---
*sql_methods_library - Samantha McGarrigle*